# 🎓 Train LLM-JEPA for Symbolic Regression

This notebook trains the LLM-JEPA model on synthetic physics equations.

**What this does:**
- Clones/pulls the repository
- Syncs to Google Drive (SymbolicRegression folder)
- Downloads AI Feynman dataset if needed
- Trains with TensorBoard monitoring
- Saves checkpoints to Drive

**Runtime:** T4 GPU or better recommended

---

In [ ]:
# @title 📦 Setup: Clone Repo & Sync to Google Drive

import os
import subprocess
from pathlib import Path
from google.colab import drive

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Define paths
DRIVE_FOLDER = "/content/drive/MyDrive/SymbolicRegression"
WORK_DIR = "/content/GSOC-LM-JEPA_for_Symbolic_Regression"
REPO_URL = "https://github.com/udohchuks/GSOC-LM-JEPA_for_Symbolic_Regression.git"

# Create Drive folder if not exists
Path(DRIVE_FOLDER).mkdir(parents=True, exist_ok=True)
print(f"✅ Drive folder ready: {DRIVE_FOLDER}")

# Clone or pull repository
if os.path.exists(WORK_DIR):
    print("📦 Repository found, pulling latest changes...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "pull"], check=True)
else:
    print("📦 Cloning repository...")
    os.chdir("/content")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(WORK_DIR)

# Sync to Drive (copy working directory)
print("🔄 Syncing to Google Drive...")
sync_target = f"{DRIVE_FOLDER}/code"
Path(sync_target).mkdir(parents=True, exist_ok=True)
subprocess.run(["rsync", "-av", "--delete", f"{WORK_DIR}/", f"{sync_target}/"], check=True)
print(f"✅ Synced to: {sync_target}")

# Install dependencies
print("📦 Installing dependencies...")
os.chdir(WORK_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed")

# Create directories in Drive
cache_dir = f"{DRIVE_FOLDER}/cache"
checkpoints_dir = f"{DRIVE_FOLDER}/checkpoints"
logs_dir = f"{DRIVE_FOLDER}/tb_logs"
data_dir = f"{DRIVE_FOLDER}/Feynman_with_units"

for d in [cache_dir, checkpoints_dir, logs_dir]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"✅ Directories ready:")
print(f"   - Cache: {cache_dir}")
print(f"   - Checkpoints: {checkpoints_dir}")
print(f"   - Logs: {logs_dir}")

# Check for synthetic data
print("\n" + "=" * 60)
print("⚠️  DEPENDENCY CHECK")
print("=" * 60)
synthetic_cache = Path(f"{cache_dir}/synthetic_1M")
if synthetic_cache.exists() and len(list(synthetic_cache.glob("*.pt"))) > 0:
    n_files = len(list(synthetic_cache.glob("*.pt")))
    print(f"✅ Synthetic data found: {n_files} files")
    print("   You can proceed with training.")
else:
    print("❌ Synthetic data NOT found!")
    print(f"   Expected location: {synthetic_cache}")
    print()
    print("📋 You must run '01_generate_synthetic_data.ipynb' first!")
    print()
    print("   Steps:")
    print("   1. Open: 01_generate_synthetic_data.ipynb")
    print("   2. Generate at least some synthetic data")
    print("   3. Return here and continue with training")
    print()
    print("⚠️  Training will fail without synthetic data!")

In [ ]:
# @title 📥 Download AI Feynman Dataset (if needed)

import tarfile
import urllib.request
from pathlib import Path

data_dir = Path(f"{DRIVE_FOLDER}/Feynman_with_units")

if data_dir.exists() and len(list(data_dir.glob("*"))) > 10:
    print(f"✅ AI Feynman dataset already exists: {data_dir}")
    print(f"   Found {len(list(data_dir.glob('*')))} files")
else:
    print("📥 Downloading AI Feynman dataset...")
    data_dir.mkdir(parents=True, exist_ok=True)
    
    # Download from Dropbox
    tar_url = "https://www.dropbox.com/s/7kgfr00qpokgz8w/Feynman_with_units.tar.gz?dl=1"
    tar_path = f"{DRIVE_FOLDER}/Feynman_with_units.tar.gz"
    
    try:
        urllib.request.urlretrieve(tar_url, tar_path)
        print("📦 Extracting...")
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(path=str(data_dir.parent))
        print(f"✅ Dataset extracted to: {data_dir}")
        
        # Cleanup
        os.remove(tar_path)
    except Exception as e:
        print(f"⚠️ Download failed: {e}")
        print("   Please download manually from the repository")

In [ ]:
# @title ⚙️ Training Configuration

# @markdown ### Select Configuration
CONFIG_FILE = "configs/small.yaml"  # @param {type: "string"}
# @markdown - `configs/small.yaml` - ~1M params, 20k-50k equations (RECOMMENDED)
# @markdown - `configs/base_config.yaml` - ~3.4M params, 100k+ equations

# @markdown ### Training Parameters
MAX_EPOCHS = 15  # @param {type: "integer"}
BATCH_SIZE = 64  # @param {type: "integer"}
LEARNING_RATE = 5e-4  # @param {type: "number"}
USE_SYNTHETIC = True  # @param {type: "boolean"}

# @markdown ### Checkpoint & Logging
EXPERIMENT_NAME = "llmjepa_small"  # @param {type: "string"}
RESUME_FROM_CHECKPOINT = False  # @param {type: "boolean"}
CHECKPOINT_PATH = ""  # @param {type: "string"}

print(f"✅ Configuration: {CONFIG_FILE}")
print(f"   Epochs: {MAX_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Checkpoints: {DRIVE_FOLDER}/checkpoints")
print()
print("📋 Note: Training command will use these parameters")

In [ ]:
# @title 🚀 Start Training

import time

print(f"🎯 Starting training...")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Epochs: {MAX_EPOCHS}")
print(f"   Using synthetic data: {USE_SYNTHETIC}")
print(f"   Config: {CONFIG_FILE}")
print("=" * 60)

start_time = time.time()

# Launch training
%cd $WORK_DIR
!python -m training.train --config {CONFIG_FILE}

elapsed = time.time() - start_time
hours = elapsed / 3600

print("\n" + "=" * 60)
print(f"✅ Training complete!")
print(f"   Time elapsed: {hours:.2f} hours ({elapsed:.0f} seconds)")
print(f"   Checkpoints saved to: {DRIVE_FOLDER}/checkpoints")
print(f"   TensorBoard logs: {DRIVE_FOLDER}/tb_logs")
print()
print("📊 Next: Run the '📊 Open TensorBoard' cell to view training progress")

---
### 📊 View Training Progress with TensorBoard

Run this cell **during or after** training to monitor progress in real-time.

In [ ]:
# @title 📊 Open TensorBoard

# Load TensorBoard extension
%load_ext tensorboard

# Open TensorBoard
log_dir = f"{DRIVE_FOLDER}/tb_logs"
print(f"📊 Opening TensorBoard...")
print(f"   Logs directory: {log_dir}")
print()
print("TensorBoard will open below. Metrics available:")
print("   - Training loss")
print("   - Validation loss")
print("   - Learning rate")
print("   - Gradient norm")
print("   - Throughput (samples/sec)")
print()
%tensorboard --logdir {log_dir}

In [ ]:
# @title 💾 List Checkpoints

from pathlib import Path

ckpt_dir = Path(f"{DRIVE_FOLDER}/checkpoints")

if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob("*.ckpt"), key=lambda x: x.stat().st_mtime, reverse=True)
    
    if ckpts:
        print(f"📁 Found {len(ckpts)} checkpoints:")
        for i, ckpt in enumerate(ckpts[:10]):  # Show latest 10
            size_mb = ckpt.stat().st_size / (1024**2)
            print(f"   {i+1}. {ckpt.name} ({size_mb:.1f} MB)")
        
        if len(ckpts) > 10:
            print(f"   ... and {len(ckpts) - 10} more")
    else:
        print("⚠️ No checkpoints found yet")
else:
    print("⚠️ Checkpoint directory not found")

---
## Next Steps

After training completes:

1. **Evaluate**: Use [`03_evaluate_model.ipynb`](03_evaluate_model.ipynb) to test on AI Feynman benchmark
2. **Inference**: Use `predict.py` to generate formulas for custom data

## Tips

- **Monitor with TensorBoard**: Keep the TensorBoard tab open to watch training in real-time
- **Early stopping**: Stop training early if validation loss plateaus
- **Checkpoints saved**: All checkpoints persist in `SymbolicRegression/checkpoints/`
- **Resume training**: Set `RESUME_FROM_CHECKPOINT = True` and provide checkpoint path